# Scoring and transforming instances with biomolecular models

In [1]:
import torch
from evedesign.system import System, Protein, SystemInstance, EntityInstance, Mutation
from evedesign.models.esm2 import ESM2
from evedesign.types import DeviceType

DEVICE: DeviceType = "cuda" if torch.cuda.is_available() else "cpu"

/Users/thomashopf/mambaforge/envs/modal/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Define system

For this example, we use E.coli beta-lactamase, with the signal peptide (1-23) removed from the target sequence.

In [2]:
target_seq = "HPETLVKVKDAEDQLGARVGYIELDLNSGKILESFRPEERFPMMSTFKVLLCGAVLSRVDAGQEQLGRRIHYSQNDLVEYSPVTEKHLTDGMTVRELCSAAITMSDNTAANLLLTTIGGPKELTAFLHNMGDHVTRLDRWEPELNEAIPNDERDTTMPAAMATTLRKLLTGELLTLASRQQLIDWMEADKVAGPLLRSALPAGWFIADKSGAGERGSRGIIAALGPDGKPSRIVVIYTTGSQATMDERNRQIAEIGASLIKHW"

system = System(
    Protein(
        id="BLAT_ECOLX", rep=target_seq, first_index=24, sequences=None, structures=None
    )
)

We also set up a corresponding instance for the reference sequence. As we specified the full target sequence on the entity above, we could also use `system.rep_to_instance()` instead of explicitly defining the instance.

In [3]:
target_instance = SystemInstance([
    EntityInstance(rep=target_seq)
])

## Set up model

Here, we use the ESM-2 LLM as an example model.

In [4]:
esm = ESM2(
    model_name="esm2_t33_650M_UR50D"
).build(system)

## Functions for scoring

Note: which of the following functions are available depends on the particular model used

### General instance scoring with score()

This is a very general function for scoring a list of instances, and can be anything from a log-likelihood for a particular sequence to a predicted Tm score from a 3D structure prediction model

In [5]:
esm.score([target_instance])

Some weights of the model checkpoint at facebook/esm2_t33_650M_UR50D were not used when initializing EsmForMaskedLM: ['esm.embeddings.position_embeddings.weight']
- This IS expected if you are initializing EsmForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing EsmForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


array([-50.48299408])

### Mutation-based scoring

The following functions are only available for models that support the notion of a mutation relative to a defined target instance.

#### Single mutation scan

This function computes a single mutation matrix relative to a specified target instance (i.e., the self-substitution is assigned a score of 0). The matrix is computed for all positions across all entities by default, unless the `entity `and `positions` attributes are specified.

Note this function may use internal optimizations to speed up the calculation or give more accurate results compared to *score()*, e.g. by computing all substitutions for a position in one model forward pass, so should be used preferentially where possible.

In [ ]:
esm.single_mutation_scan(target_instance)

#### Arbitrary mutant scoring

This function allows to score arbitrary single and higher-order mutants relative to the target instance. Individual mutants are specified as a list of *Mutation* objects.

Note this function may use internal optimizations to speed up the calculation or give more accurate results compared to *score()*, so should be used preferentially where possible.

In [28]:
# single mutant
single_mutant = [Mutation(entity=0, pos=180, ref="M", to="T")]
double_mutant = [Mutation(entity=0, pos=180, ref="M", to="T"), Mutation(entity=0, pos=236, ref="G", to="S"),]

esm.score_mutants(
    target_instance,
    [
        single_mutant,
        double_mutant,
    ]
)

array([ 5.10602111, -1.06581936])

#### Conditional mutation scoring with score_conditional()

This function is typically only of relevance for applications where the conditional likelihood P(x_i | x_\i) needs to be computed, e.g. for Gibbs sampling.

## Transform

This operation transforms instances from one representation level to another. In the case of ESM-2 shown here, it maps sequences to embeddings.

The *transform()* method may also set the *score* attribute of the instance if a score can be computed in the same model pass to make computations more efficient.

In [30]:
instances_transformed = esm.transform([target_instance])

In [31]:
# indices: first instance, first entity
instances_transformed[0][0].embedding

array([[ 9.45182443e-02,  8.84119943e-02,  2.38465648e-02, ...,
        -8.20149183e-02,  1.21000335e-01, -6.58196956e-02],
       [ 1.52301773e-01,  1.07781410e-01, -1.68123171e-02, ...,
        -1.46298438e-01,  1.65676892e-01,  1.86714903e-02],
       [ 1.56093135e-01,  6.56763762e-02, -1.77681074e-03, ...,
         6.84123337e-02,  1.25577629e-01,  2.97634639e-02],
       ...,
       [-1.28520653e-04,  1.62136436e-01,  2.56760214e-02, ...,
        -1.99570388e-01, -2.18329549e-01, -6.81631416e-02],
       [ 6.90786093e-02, -2.48125680e-02, -1.29462764e-01, ...,
        -2.66222972e-02,  5.01280427e-02, -2.78342366e-02],
       [-5.63203879e-02,  9.42330286e-02,  2.08682358e-01, ...,
        -2.98920780e-01, -6.42098263e-02,  1.33362547e-01]],
      shape=(263, 1280), dtype=float32)

In [33]:
instances_transformed[0].score

-50.482994079589844